In [1]:
# 1. Hubungkan ke Google Drive
from google.colab import drive
import pandas as pd
import numpy as np

# Mount drive
drive.mount('/content/drive')

# 2. Load Data
# Sesuaikan path ini dengan lokasi file Anda di Drive
file_path = '/content/drive/My Drive/tugas_monte_carlo/crop_production.csv'
df = pd.read_csv(file_path)

# --- TAHAP PRE-PROCESSING ---
# Kita ambil contoh satu tanaman spesifik di satu daerah agar datanya linier seperti contoh tugas
# Misal: Tanaman 'Rice' di state 'Andhra Pradesh' (atau bisa di-agregat per tahun)
target_crop = 'Rice'
filtered_df = df[df['Crop'] == target_crop]

# Kita group berdasarkan Tahun untuk melihat total produksi per tahun
data_historis = filtered_df.groupby('Crop_Year')['Production'].sum().reset_index()

# Kita ambil 5-10 tahun terakhir saja agar mirip contoh tabel di PDF
data_simulasi = data_historis.tail(10).reset_index(drop=True)
data_simulasi.columns = ['Tahun', 'Jumlah_Produksi'] # Rename kolom

# --- TAHAP 1: MENGHITUNG INTERVAL (Sesuai PDF Hal 1) ---
total_produksi = data_simulasi['Jumlah_Produksi'].sum()

# Hitung Probabilitas
data_simulasi['Probabilitas'] = data_simulasi['Jumlah_Produksi'] / total_produksi

# Hitung Probabilitas Kumulatif
data_simulasi['Kumulatif'] = data_simulasi['Probabilitas'].cumsum()

# Tentukan Interval Batas Bawah & Atas
# Logic: Batas bawah dimulai dari 0 atau batas atas sebelumnya + 1
batas_bawah = [0]
batas_atas = []

for i in range(len(data_simulasi)):
    # Mengubah kumulatif menjadi skala integer (misal 0-100 atau 0-1000)
    # Di PDF contohnya pakai skala besar, kita pakai skala 1000 biar presisi
    top_limit = round(data_simulasi.loc[i, 'Kumulatif'] * 1000)
    batas_atas.append(top_limit)
    if i < len(data_simulasi) - 1:
        batas_bawah.append(top_limit + 1)

data_simulasi['Batas_Bawah'] = batas_bawah
data_simulasi['Batas_Atas'] = batas_atas

print("--- TABEL DISTRIBUSI FREKUENSI ---")
print(data_simulasi[['Tahun', 'Jumlah_Produksi', 'Probabilitas', 'Kumulatif', 'Batas_Bawah', 'Batas_Atas']])


# --- TAHAP 2: GENERATE BILANGAN ACAK (Metode LCM) ---
# Menggunakan Linear Congruential Method sesuai PDF (Zi = (a.Zi-1 + c) mod m)

def lcm_random(n, seed=123, a=1664525, c=1013904223, m=2**32):
    """
    Generate n random numbers using Linear Congruential Method.
    Ini manual generator sesuai teori, bukan pakai random.randint bawaan python
    agar sesuai kaidah kuliah simulasi.
    """
    random_numbers = []
    zi = seed
    for _ in range(n):
        zi = (a * zi + c) % m
        # Kita scale down hasil Zi ke range interval kita (0-1000)
        # Agar bisa dicocokkan dengan tabel interval
        scaled_num = int((zi / m) * 1000)
        random_numbers.append(scaled_num)
    return random_numbers

# Prediksi untuk 5 tahun ke depan (misalnya)
n_prediksi = 5
angka_acak = lcm_random(n_prediksi)

hasil_prediksi = []

print("\n--- HASIL SIMULASI MONTE CARLO ---")
for acak in angka_acak:
    # Cari angka acak masuk ke interval tahun berapa
    prediksi_row = data_simulasi[
        (data_simulasi['Batas_Bawah'] <= acak) &
        (data_simulasi['Batas_Atas'] >= acak)
    ]

    if not prediksi_row.empty:
        nilai_prediksi = prediksi_row['Jumlah_Produksi'].values[0]
        tahun_referensi = prediksi_row['Tahun'].values[0]
        hasil_prediksi.append({
            'Angka_Acak': acak,
            'Prediksi_Produksi': nilai_prediksi,
            'Berdasarkan_Pola_Tahun': tahun_referensi
        })

df_hasil = pd.DataFrame(hasil_prediksi)
print(df_hasil)

# Hitung Rata-rata prediksi (Opsional, untuk kesimpulan)
rata_rata_prediksi = df_hasil['Prediksi_Produksi'].mean()
print(f"\nRata-rata prediksi produksi {target_crop} untuk periode mendatang: {rata_rata_prediksi:.2f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- TABEL DISTRIBUSI FREKUENSI ---
   Tahun  Jumlah_Produksi  Probabilitas  Kumulatif  Batas_Bawah  Batas_Atas
0   2006     9.167629e+07      0.104564   0.104564            0         105
1   2007     9.206481e+07      0.105008   0.209572          106         210
2   2008     9.526127e+07      0.108653   0.318226          211         318
3   2009     8.735634e+07      0.099637   0.417863          319         418
4   2010     9.517621e+07      0.108556   0.526419          419         526
5   2011     1.020939e+08      0.116447   0.642866          527         643
6   2012     1.003305e+08      0.114435   0.757301          644         757
7   2013     1.028728e+08      0.117335   0.874636          758         875
8   2014     1.040182e+08      0.118641   0.993278          876         993
9   2015     5.893687e+06      0.006722   1.000000          994        1000
